# Model Predictions Evaluation (Train/Test, No Leakage)

- 입력: `../outputs/splits/{train,test}.csv` + `../data/DPQ_L.xpt`
- 출력:
  - 베이스라인 설정: `../outputs/config/baseline.json`
  - 베이스라인 예측: `../outputs/predictions/test_predictions.baseline.csv`
  - 모델 예측: `../outputs/predictions/test_predictions.model.csv`
- 프롬프트: PHQ-9 9문항(Q1~Q9)만 포함(총점/중증도 미포함)
- 평가: accuracy, macro-F1, 혼동행렬(riskLevel), needsAttention F1


In [1]:
import os, json, time, requests
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

# 경로
data_dir = Path('../outputs/splits')
config_dir = Path('../outputs/config'); config_dir.mkdir(parents=True, exist_ok=True)
pred_dir = Path('../outputs/predictions'); pred_dir.mkdir(parents=True, exist_ok=True)

# 데이터 로드
train_df = pd.read_csv(data_dir / 'train.csv')
test_df = pd.read_csv(data_dir / 'test.csv')

# DPQ 9문항 조인
dpq_items = pd.read_sas('../data/DPQ_L.xpt', format='xport')[
    ['SEQN','DPQ010','DPQ020','DPQ030','DPQ040','DPQ050','DPQ060','DPQ070','DPQ080','DPQ090']
]
train_df = train_df.merge(dpq_items, on='SEQN', how='left')
test_df = test_df.merge(dpq_items, on='SEQN', how='left')

print('train/test =', len(train_df), len(test_df))


train/test = 93 24


In [2]:
# 베이스라인 임계값 선택(train에서만)
candidates = [8,9,10,12]
best = None
for th in candidates:
    y_true = train_df['needsAttention'].astype(bool)
    y_pred = (train_df['phq9_sum'] >= th).astype(bool)
    f1 = f1_score(y_true, y_pred)
    if not best or f1 > best['f1']:
        best = {'threshold': th, 'f1': f1}
print('selected threshold on train =', best)

# 설정 저장
with open(config_dir / 'baseline.json', 'w', encoding='utf-8') as f:
    json.dump(best, f, ensure_ascii=False, indent=2)


selected threshold on train = {'threshold': 10, 'f1': 1.0}


In [3]:
# 베이스라인 예측(test에 적용)
with open(config_dir / 'baseline.json', 'r', encoding='utf-8') as f:
    cfg = json.load(f)
th = cfg['threshold']

baseline = pd.DataFrame({
    'SEQN': test_df['SEQN'],
    'pred_needsAttention': (test_df['phq9_sum'] >= th).astype(bool),
    'pred_riskLevel': test_df['phq9_severity']  # riskLevel은 규칙 기반 매핑 가능하나, 여기선 비교 목적으로 보류/또는 동일매핑 금지
})

# 저장
baseline_path = pred_dir / 'test_predictions.baseline.csv'
baseline.to_csv(baseline_path, index=False)
print('saved baseline:', baseline_path)


saved baseline: ..\outputs\predictions\test_predictions.baseline.csv


In [9]:
# OpenAI 호출 설정
auth_key = os.getenv('OPENAI_API_KEY')
api_url = os.getenv('OPENAI_API_URL', 'https://api.openai.com/v1/chat/completions')

headers = {
    'Content-Type': 'application/json',
    'Authorization': f'Bearer {auth_key}',
}
SYSTEM_PROMPT = '당신은 노인 요양원의 의료진을 도와주는 AI 의료 분석 전문가입니다.'

def build_prompt(row):
    return (
        '지난 2주의 PHQ-9 9문항 응답을 바탕으로, 다음 두 값을 JSON으로만 출력하세요.\n'
        '필드: riskLevel{none|mild|moderate|moderately_severe|severe}, needsAttention{true|false}.\n'
        f"Q1={row['DPQ010']}, Q2={row['DPQ020']}, Q3={row['DPQ030']}, Q4={row['DPQ040']}, "
        f"Q5={row['DPQ050']}, Q6={row['DPQ060']}, Q7={row['DPQ070']}, Q8={row['DPQ080']}, Q9={row['DPQ090']}"
    )

def call_openai(prompt: str):
    body = {
        'model':'gpt-5-mini',
        'messages':[{'role':'system','content':SYSTEM_PROMPT},{'role':'user','content':prompt}],
        'response_format':{'type':'json_object'}
    }
    r = requests.post(api_url, headers=headers, json=body, timeout=120)
    r.raise_for_status()
    content = r.json()['choices'][0]['message']['content']
    try:
        return json.loads(content)
    except Exception:
        if '```json' in content:
            s = content.find('```json')+7; e = content.rfind('```'); return json.loads(content[s:e].strip())
        raise


In [10]:
# 모델 예측(test) 생성 및 저장
pred_rows = []
for _, row in test_df.iterrows():
    prompt = build_prompt(row)
    try:
        resp = call_openai(prompt)
        risk = str(resp.get('riskLevel','')).strip().lower()
        attn = resp.get('needsAttention', None)
        pred_rows.append({'SEQN': row['SEQN'], 'pred_riskLevel': risk, 'pred_needsAttention': bool(attn) if isinstance(attn,bool) else str(attn).lower()=='true'})
    except Exception as e:
        pred_rows.append({'SEQN': row['SEQN'], 'pred_riskLevel': '', 'pred_needsAttention': ''})
        print('error on', row['SEQN'], e)
    time.sleep(0.5)
model_path = pred_dir / 'test_predictions.model.csv'
pd.DataFrame(pred_rows).to_csv(model_path, index=False)
print('saved model predictions:', model_path)


saved model predictions: ..\outputs\predictions\test_predictions.model.csv


In [11]:
# 평가: 베이스라인 vs 모델
labels = ['none','mild','moderate','moderately_severe','severe']

def eval_file(path, name):
    preds = pd.read_csv(path)
    merged = test_df.merge(preds, on='SEQN', how='inner')
    # 경고(동일 여부)
    equal_rate = (merged['pred_riskLevel'].astype(str)==merged['phq9_severity'].astype(str)).mean()
    if equal_rate > 0.99:
        print(f'[경고] {name}: riskLevel이 라벨과 거의 동일합니다({equal_rate:.3f}).')
    # riskLevel
    y_true_rl = merged['phq9_severity']
    y_pred_rl = merged['pred_riskLevel']
    print(f'[{name}][riskLevel] acc=', accuracy_score(y_true_rl, y_pred_rl), 'macro-F1=', f1_score(y_true_rl, y_pred_rl, average='macro'))
    print(confusion_matrix(y_true_rl, y_pred_rl, labels=labels))
    # needsAttention
    y_true_na = merged['needsAttention'].astype(bool)
    y_pred_na = merged['pred_needsAttention'].astype(bool)
    print(f'[{name}][needsAttention] acc=', accuracy_score(y_true_na, y_pred_na), 'F1=', f1_score(y_true_na, y_pred_na))

print('--- Baseline (rules) ---')
eval_file(pred_dir / 'test_predictions.baseline.csv', 'baseline')
print('--- Model (OpenAI) ---')
eval_file(pred_dir / 'test_predictions.model.csv', 'model')


--- Baseline (rules) ---
[경고] baseline: riskLevel이 라벨과 거의 동일합니다(1.000).
[baseline][riskLevel] acc= 1.0 macro-F1= 1.0
[[ 0  0  0  0  0]
 [ 0  0  0  0  0]
 [ 0  0  4  0  0]
 [ 0  0  0 10  0]
 [ 0  0  0  0 10]]
[baseline][needsAttention] acc= 1.0 F1= 1.0
--- Model (OpenAI) ---
[경고] model: riskLevel이 라벨과 거의 동일합니다(1.000).
[model][riskLevel] acc= 1.0 macro-F1= 1.0
[[ 0  0  0  0  0]
 [ 0  0  0  0  0]
 [ 0  0  4  0  0]
 [ 0  0  0 10  0]
 [ 0  0  0  0 10]]
[model][needsAttention] acc= 1.0 F1= 1.0


# Model Predictions Evaluation (No Leakage)

- 입력: `../outputs/splits/test.csv` + `../data/DPQ_L.xpt`
- 출력: `../outputs/predictions/test_predictions.model.csv`
- 프롬프트: PHQ-9 9문항(Q1~Q9)만 포함(총점/중증도 미포함)


In [ ]:
import pandas as pd
from pathlib import Path

# 테스트셋 로드
data_dir = Path('../outputs/splits')
test_df = pd.read_csv(data_dir / 'test.csv')

# DPQ 9문항 조인
dpq_items = pd.read_sas('../data/DPQ_L.xpt', format='xport')[
    ['SEQN','DPQ010','DPQ020','DPQ030','DPQ040','DPQ050','DPQ060','DPQ070','DPQ080','DPQ090']
]

test_df = test_df.merge(dpq_items, on='SEQN', how='left')
print('merged test size =', len(test_df))
test_df.head(3)


merged test size = 24


,SEQN,phq9_sum,phq9_severity,needsAttention,DPQ010,DPQ020,DPQ030,DPQ040,DPQ050,DPQ060,DPQ070,DPQ080,DPQ090
0,135067.0,19.0,moderately_severe,True,2.0,1.0,3.0,3.0,2.0,3.0,1.0,2.0,2.0
1,132368.0,14.0,moderate,True,2.0,2.0,3.0,2.0,1.0,1.0,1.0,1.0,1.0
2,135028.0,15.0,moderately_severe,True,1.0,3.0,3.0,1.0,1.0,1.0,2.0,2.0,1.0


In [ ]:
def build_prompt(row):
    return (
        '지난 2주의 PHQ-9 9문항 응답을 바탕으로, 다음 두 값을 JSON으로만 출력하세요.\n'
        '필드: riskLevel{none|mild|moderate|moderately_severe|severe}, needsAttention{true|false}.\n'
        f"Q1={row['DPQ010']}, Q2={row['DPQ020']}, Q3={row['DPQ030']}, Q4={row['DPQ040']}, "
        f"Q5={row['DPQ050']}, Q6={row['DPQ060']}, Q7={row['DPQ070']}, Q8={row['DPQ080']}, Q9={row['DPQ090']}"
    )


In [ ]:
# 나머지 셀(호출/저장/평가)은 기존 것 재사용
# - call_openai, 배치 호출 저장, 평가 셀을 위 프롬프트에 맞춰 실행하세요.


# Model Predictions Evaluation (Separated)

- 목적: 베이스라인과 분리된 모델 예측 파일 생성/평가
- 입력: `../outputs/splits/test.csv`
- 출력: `../outputs/predictions/test_predictions.model.csv`
- 평가: accuracy, macro-F1, 혼동행렬(riskLevel)


In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "your api key"
os.environ["OPENAI_API_URL"] = "https://api.openai.com/v1/chat/completions"

In [ ]:
import json, time, requests
import pandas as pd
from pathlib import Path

# 경로 설정
data_dir = Path('../outputs/splits')
pred_dir = Path('../outputs/predictions')
pred_dir.mkdir(parents=True, exist_ok=True)

# 환경변수
auth_key = os.getenv('OPENAI_API_KEY')
api_url = os.getenv('OPENAI_API_URL', 'https://api.openai.com/v1/chat/completions')

headers = {
    'Content-Type': 'application/json',
    'Authorization': f'Bearer {auth_key}',
}

SYSTEM_PROMPT = '당신은 노인 요양원의 의료진을 도와주는 AI 의료 분석 전문가입니다.'

# 입력 로드
test_df = pd.read_csv(data_dir / 'test.csv')
print('test size =', len(test_df))


test size = 24


In [ ]:
def build_prompt(row):
    return (
        '지난 2주의 우울 증상 설문(PHQ-9) 응답을 기반으로, 다음 두 값을 JSON으로만 출력하세요.\n'
        '필드: riskLevel{none|mild|moderate|moderately_severe|severe}, needsAttention{true|false}.\n'
        f"총점(0-27): {int(row['phq9_sum']) if pd.notna(row['phq9_sum']) else 'NA'}"
    )

def call_openai(prompt: str):
    body = {
        'model': 'gpt-5-mini',
        'messages': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': prompt}
        ],
        'response_format': {'type': 'json_object'},
    }
    r = requests.post(api_url, headers=headers, json=body, timeout=120)
    r.raise_for_status()
    content = r.json()['choices'][0]['message']['content']
    try:
        return json.loads(content)
    except Exception:
        if '```json' in content:
            start = content.find('```json') + 7
            end = content.rfind('```')
            return json.loads(content[start:end].strip())
        raise


In [ ]:
# 배치 호출 및 저장(모델 예측 파일명 분리)
pred_rows = []
for _, row in test_df.iterrows():
    prompt = build_prompt(row)
    try:
        resp = call_openai(prompt)
        risk = str(resp.get('riskLevel', '')).strip().lower()
        attn = resp.get('needsAttention', None)
        pred_rows.append({'SEQN': row['SEQN'], 'pred_riskLevel': risk, 'pred_needsAttention': bool(attn) if isinstance(attn, bool) else str(attn).lower()=='true'})
    except Exception as e:
        pred_rows.append({'SEQN': row['SEQN'], 'pred_riskLevel': '', 'pred_needsAttention': ''})
        print('error on SEQN', row['SEQN'], e)
    time.sleep(0.5)

model_pred_path = pred_dir / 'test_predictions.model.csv'
pd.DataFrame(pred_rows).to_csv(model_pred_path, index=False)
print('saved model predictions:', model_pred_path)


saved model predictions: ..\outputs\predictions\test_predictions.model.csv


In [ ]:
# 평가
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

preds = pd.read_csv(model_pred_path)
merged = test_df.merge(preds, on='SEQN', how='inner')

# 베이스라인 파일과 혼동 방지 경고
equal_rate = (merged['pred_riskLevel'].astype(str)==merged['phq9_severity'].astype(str)).mean()
if equal_rate > 0.99:
    print('[경고] 예측이 라벨과 거의 동일합니다. 베이스라인 파일이 아닌지 확인하세요.')

# riskLevel
y_true_rl = merged['phq9_severity']
y_pred_rl = merged['pred_riskLevel']
labels = ['none','mild','moderate','moderately_severe','severe']
print('[riskLevel] accuracy=', accuracy_score(y_true_rl, y_pred_rl),
      ' macro-F1=', f1_score(y_true_rl, y_pred_rl, average='macro'))
print('CM:\n', confusion_matrix(y_true_rl, y_pred_rl, labels=labels))
print(classification_report(y_true_rl, y_pred_rl, labels=labels, digits=3, zero_division=0))

# needsAttention
y_true_na = merged['needsAttention'].astype(bool)
y_pred_na = merged['pred_needsAttention'].astype(bool)
print('[needsAttention] accuracy=', accuracy_score(y_true_na, y_pred_na),
      ' F1=', f1_score(y_true_na, y_pred_na))


[경고] 예측이 라벨과 거의 동일합니다. 베이스라인 파일이 아닌지 확인하세요.
[riskLevel] accuracy= 1.0  macro-F1= 1.0
CM:
 [[ 0  0  0  0  0]
 [ 0  0  0  0  0]
 [ 0  0  4  0  0]
 [ 0  0  0 10  0]
 [ 0  0  0  0 10]]
                   precision    recall  f1-score   support

             none      0.000     0.000     0.000         0
             mild      0.000     0.000     0.000         0
         moderate      1.000     1.000     1.000         4
moderately_severe      1.000     1.000     1.000        10
           severe      1.000     1.000     1.000        10

         accuracy                          1.000        24
        macro avg      0.600     0.600     0.600        24
     weighted avg      1.000     1.000     1.000        24

[needsAttention] accuracy= 1.0  F1= 1.0
